# polars-usaddress vs. Python `usaddress`: throughput benchmark

Compares the native Polars plugin (`polars_usaddress.tag_address`, one vectorized
call over a whole column) against the reference implementation
(`usaddress.tag`, called once per address, as a typical Python caller would use it).

**Prerequisite:** build the plugin into *this kernel's* environment first --
this notebook does not do that for you:

```bash
maturin develop --release
```

(`--release` matters: debug builds are ~20x slower and would make this an unfair fight.)

In [10]:
import random
import time

import polars as pl

try:
    import polars_usaddress as plua
except ImportError as exc:
    raise SystemExit(
        "polars_usaddress isn't importable in this kernel.\n"
        "Build it into this kernel's environment first:\n\n"
        "    maturin develop --release\n"
    ) from exc

import usaddress
from usaddress import RepeatedLabelError

print("polars version:              ", pl.__version__)
print("polars_usaddress pinned to:  ", plua.UPSTREAM_VERSION)

polars version:               1.44.2
polars_usaddress pinned to:   0.5.16


## Synthetic benchmark corpus

Exercises the same tokenizer/feature-encoder edge cases the parity suite covers --
directionals, abbreviations, fractions, punctuation tails, PO boxes -- not just
clean street addresses.
Seeded, so the corpus is identical on every run.

In [11]:
random.seed(1234)

NUM = ["1", "12", "123", "1200", "10000", "123A", "1/2", "\u00bd", "170th", "(45)"]
STREETS = [
    "Main",
    "Oak",
    "Elm",
    "Caf\u00e9",
    "\u00d1andu",
    "St",
    "Ave.",
    "Blvd",
    "N",
    "SW",
    "O'Hare",
]
TAIL = ["", ",", ".", ";", ")", ",,", ".\n"]
EXTRA = ["#", "&", "Apt 3B", "PO Box 12", "Suite 100", "and", "Chicago, IL 60601", ""]


def make_corpus(n):
    out = []
    for _ in range(n):
        parts = [random.choice(NUM)] if random.random() < 0.8 else []
        for _ in range(random.randint(1, 4)):
            parts.append(random.choice(STREETS) + random.choice(TAIL))
        if random.random() < 0.5:
            parts.append(random.choice(EXTRA))
        out.append(" ".join(parts))
    return out


CORPUS_SIZE = 1_000_000
corpus = make_corpus(CORPUS_SIZE)
print(f"generated {len(corpus)} synthetic addresses")
corpus[:5]

generated 1000000 synthetic addresses


["Oak Oak.\n SW O'Hare,, and",
 '1 St,,',
 '123 Elm,, #',
 "1200 O'Hare)",
 '12 Ñandu.\n']

## Correctness spot-check

Before trusting the timing numbers, confirm the two engines actually agree on a sample.
A `RepeatedLabelError` from upstream is expected to come back as an all-null row from the
plugin (see `tag_address`'s docstring), so that case is handled specially rather than
counted as a mismatch.

In [12]:
def python_tag(addr):
    try:
        tagged, _addr_type = usaddress.tag(addr)
        return dict(tagged)
    except RepeatedLabelError:
        return None


SAMPLE_N = 300
sample = corpus[:SAMPLE_N]

rust_rows = (
    pl.DataFrame({"address": sample})
    .with_columns(parsed=plua.tag_address("address"))
    .unnest("parsed")
    .to_dicts()
)

mismatches = []
for addr, row in zip(sample, rust_rows):
    want = python_tag(addr)
    got = {k: v for k, v in row.items() if k in plua.LABELS and v is not None}
    if want is None:
        if got:
            mismatches.append(
                (addr, "expected all-null (RepeatedLabelError upstream)", got)
            )
        continue
    if got != want:
        mismatches.append((addr, want, got))

print(f"checked {SAMPLE_N} addresses, {len(mismatches)} mismatches")
for addr, want, got in mismatches:
    print("\n ", repr(addr))
    print("  want:", want)
    print("  got :", got)

checked 300 addresses, 0 mismatches


## Throughput: one batch at a fixed size

`time_python` calls `usaddress.tag` in a loop, the way a Python caller normally would.
`time_polars` builds a one-column DataFrame and makes a single vectorized call --
the way the plugin is meant to be used. Both include only the tagging work itself;
corpus generation happened above.

In [13]:
def time_python(addresses):
    start = time.perf_counter()
    for addr in addresses:
        python_tag(addr)
    return time.perf_counter() - start


def time_polars(addresses):
    df = pl.DataFrame({"address": addresses})
    start = time.perf_counter()
    df.with_columns(parsed=plua.tag_address("address")).unnest("parsed")
    # df.with_columns(parsed=plua.tag_address_with_confidence("address")).unnest("parsed")
    return time.perf_counter() - start


BENCH_N = 20_000  # keep the pure-Python side finishing in a reasonable time
bench_subset = corpus[:BENCH_N]  # owned by this section -- don't reuse this name below

# Warm-up: make sure lazy one-time costs (CRF model load, JIT-ish caches) aren't
# what we end up timing.
python_tag(bench_subset[0])
time_polars(bench_subset[:10])

bench_py_time = time_python(bench_subset)
bench_rs_time = time_polars(bench_subset)

print(f"{'engine':<24}{'total s':>10}{'addrs/s':>14}")
print(
    f"{'python usaddress':<24}{bench_py_time:>10.2f}{BENCH_N / bench_py_time:>14,.0f}"
)
print(
    f"{'polars_usaddress':<24}{bench_rs_time:>10.2f}{BENCH_N / bench_rs_time:>14,.0f}"
)
print(f"\nspeedup: {bench_py_time / bench_rs_time:,.1f}x")

engine                     total s       addrs/s
python usaddress              0.67        30,034
polars_usaddress              0.04       527,662

speedup: 17.6x


## Scaling sweep

Same comparison across corpus sizes, to see how much of the gap is fixed per-call overhead
(FFI, Python-object marshalling) versus per-address work.

In [14]:
sizes = [
    n for n in [100, 500, 1_000, 5_000, 20_000, 50_000, 1_000_000] if n <= len(corpus)
]

rows = []
for n in sizes:
    sweep_subset = corpus[:n]
    py_t = time_python(sweep_subset)
    rs_t = time_polars(sweep_subset)
    rows.append(
        {
            "n": n,
            "python_s": py_t,
            "polars_s": rs_t,
            "python_addrs_per_s": n / py_t,
            "polars_addrs_per_s": n / rs_t,
            "speedup": py_t / rs_t,
        }
    )

results = pl.DataFrame(rows)
results

n,python_s,polars_s,python_addrs_per_s,polars_addrs_per_s,speedup
i64,f64,f64,f64,f64,f64
100,0.003344,0.0015,29903.555073,66651.825468,2.228893
500,0.019988,0.002905,25015.321878,172112.122214,6.880268
1000,0.033024,0.003163,30280.817028,316126.464908,10.439826
5000,0.165534,0.009564,30205.30534,522784.736224,17.307712
20000,0.659141,0.031833,30342.51198,628272.257894,20.706007
50000,1.639723,0.086052,30492.960624,581041.487804,19.054938
1000000,32.927551,2.415035,30369.704958,414072.752593,13.634402


In [15]:
# Interactive chart in a browser tab -- avoids matplotlib's inline PNG (or
# Plotly's own "notebook" renderer) getting base64-embedded into this cell's
# saved output, which would otherwise bloat every commit that touches this
# file. `plotly` is a dev dependency (`uv sync` picks it up); this degrades
# gracefully without it, same as the old matplotlib cell did.
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Throughput", "polars_usaddress speedup over python usaddress"),
)

fig.add_trace(
    go.Scatter(
        x=results["n"].to_list(),
        y=results["python_addrs_per_s"].to_list(),
        mode="lines+markers",
        name="python usaddress",
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=results["n"].to_list(),
        y=results["polars_addrs_per_s"].to_list(),
        mode="lines+markers",
        name="polars_usaddress",
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=results["n"].to_list(),
        y=results["speedup"].to_list(),
        mode="lines+markers",
        name="speedup",
        line={"color": "green"},
        showlegend=False,
    ),
    row=1,
    col=2,
)

fig.update_xaxes(title_text="addresses", row=1, col=1)
fig.update_yaxes(title_text="addresses / second", row=1, col=1)
fig.update_xaxes(title_text="addresses", row=1, col=2)
fig.update_yaxes(title_text="speedup (x)", row=1, col=2)

fig.update_layout(width=1100, height=420)
fig.show(
    renderer="browser"
)  # opens in the default browser; nothing gets embedded in this notebook's output

## Isolating the CRF tagging cost

The scaling sweep above holds at ~2x from N=500 up, instead of growing with N --
that rules out fixed per-call/FFI overhead as the explanation (if it were that, the
speedup would keep climbing as N grows and overhead gets amortized). So the gap is a
genuine, roughly-constant *per-address* cost difference.

`usaddress.tag` does two things: build a Python feature dict per token
(`tokens2features`), then hand that to `pycrfsuite`'s C++ tagger. Only the first part
is pure Python -- the actual CRF inference is already native code on both sides of this
comparison (CRFsuite vs. `crfs`, a from-scratch Rust port of the same format).
If CRF tagging itself is most of `usaddress.tag`'s time, then the win available to
the Rust plugin is capped by how `crfs`'s inference compares to CRFsuite's, not by
skipping Python's feature-dict construction.

This isolates tagging-only time by pre-encoding every address's feature sequence
once (outside the timed region), then timing just `tagger.tag(seq)` in a loop --
same corpus, same `usaddress.tag` cell above, so it's directly comparable to `bench_py_time`.

In [16]:
import pycrfsuite

tagger_only = pycrfsuite.Tagger()
tagger_only.open(usaddress.MODEL_PATH)


def encode(tokens):
    return pycrfsuite.ItemSequence(usaddress.tokens2features(tokens))


# Pre-encode once, outside the timed region -- we're isolating tagging cost only.
# Uses bench_subset/bench_py_time from the "Throughput" section above, not the
# scaling sweep's sweep_subset -- so re-run that cell first if you've restarted
# the kernel or want a fresh measurement.
encoded_seqs = [encode(usaddress.tokenize(a)) for a in bench_subset]
encoded_seqs = [seq for seq in encoded_seqs if len(seq) > 0]

start = time.perf_counter()
for seq in encoded_seqs:
    tagger_only.tag(seq)
crf_only_time = time.perf_counter() - start

feature_time = bench_py_time - crf_only_time
print(f"full usaddress.tag() loop:      {bench_py_time:>8.2f} s  (n={BENCH_N})")
print(
    f"pycrfsuite tagging only:        {crf_only_time:>8.2f} s  ({100 * crf_only_time / bench_py_time:5.1f}% of total)"
)
print(
    f"tokenize + feature extraction:  {feature_time:>8.2f} s  ({100 * feature_time / bench_py_time:5.1f}% of total, by subtraction)"
)
print()
print(
    f"polars_usaddress total:         {bench_rs_time:>8.2f} s  (for comparison -- does tokenize + features + tagging, all in Rust)"
)

full usaddress.tag() loop:          0.67 s  (n=20000)
pycrfsuite tagging only:            0.15 s  ( 23.0% of total)
tokenize + feature extraction:      0.51 s  ( 77.0% of total, by subtraction)

polars_usaddress total:             0.04 s  (for comparison -- does tokenize + features + tagging, all in Rust)


### Cross-checking against the Rust-side split

`examples/bench_split.rs` measures the same feature-extraction/tagging split
directly in Rust, via `cargo run --release --no-default-features --features
bench-timing --example bench_split`. By default it cycles the 50 addresses in
`tests/fixtures.json` (several of which are longer, multi-clause
`RepeatedLabelError` cases) rather than this notebook's synthetic `corpus` --
so its raw totals aren't comparable to `bench_py_time`/`bench_rs_time` above. Export the
exact same `bench_subset` used in the cells above so the comparison is apples-to-apples:

In [17]:
import json
from pathlib import Path

corpus_path = Path("bench_corpus.json").resolve()
corpus_path.write_text(json.dumps(bench_subset))
print(f"wrote {len(bench_subset)} addresses to {corpus_path}")
print()
print("then, from the project root:")
print(
    "cargo run --release --no-default-features --features bench-timing "
    f"--example bench_split -- {corpus_path}"
)

wrote 20000 addresses to /Users/zackery/Code/polars-usaddress/tools/bench_corpus.json

then, from the project root:
cargo run --release --no-default-features --features bench-timing --example bench_split -- /Users/zackery/Code/polars-usaddress/tools/bench_corpus.json


## Python multiprocessing

`pycrfsuite` is a C extension and doesn't release the GIL during tagging, so Python
*threads* can't use extra cores here -- but separate *processes* can. The worker
function lives in `mp_worker.py`, not a cell in this notebook: `ProcessPoolExecutor`'s
default "spawn" start method (macOS/Windows) pickles a *reference* to the function and
re-imports it in each worker, and a function defined in a Jupyter cell lives in the
kernel's `__main__`, which a spawned worker can't re-import.

Each worker imports `usaddress` once when it starts (which is when `usaddress` loads its
own `pycrfsuite.Tagger`) and is then reused for many tasks -- the same "one per worker,
not one per row" shape as the Rust side's `map_init`. The pool is created once and kept
open across the warm-up and the timed run, so process-spawn/import cost lands in the
warm-up, not the measurement -- same reasoning as every other warm-up call above.

In [18]:
import os
import sys
from concurrent.futures import ProcessPoolExecutor

# mp_worker.py sits next to this notebook; add its directory to sys.path in case
# the kernel's cwd isn't already there.
for candidate in (os.getcwd(), os.path.join(os.getcwd(), "tools")):
    if (
        os.path.exists(os.path.join(candidate, "mp_worker.py"))
        and candidate not in sys.path
    ):
        sys.path.insert(0, candidate)

import mp_worker

N_WORKERS = os.cpu_count()
print(f"workers available: {N_WORKERS}")

# A handful of chunks per worker, so one short/slow chunk doesn't stall the whole
# pool, without paying IPC overhead per single address.
chunksize = max(1, len(bench_subset) // (N_WORKERS * 4))

with ProcessPoolExecutor(max_workers=N_WORKERS) as pool:
    # Warm-up: absorb worker spawn + usaddress/pycrfsuite import+model-load cost
    # here, not in the timed region below.
    list(pool.map(mp_worker.tag_one, bench_subset[:200], chunksize=1))

    start = time.perf_counter()
    list(pool.map(mp_worker.tag_one, bench_subset, chunksize=chunksize))
    mp_time = time.perf_counter() - start

mp_label = f"python usaddress (mp, {N_WORKERS} workers)"
print(f"{'engine':<32}{'total s':>10}{'addrs/s':>14}")
print(
    f"{'python usaddress (1 core)':<32}{bench_py_time:>10.2f}{BENCH_N / bench_py_time:>14,.0f}"
)
print(f"{mp_label:<32}{mp_time:>10.2f}{BENCH_N / mp_time:>14,.0f}")
print(
    f"{'polars_usaddress':<32}{bench_rs_time:>10.2f}{BENCH_N / bench_rs_time:>14,.0f}"
)
print()
print(
    f"multiprocessing speedup over single-core python: {bench_py_time / mp_time:,.1f}x"
)
print(
    f"polars_usaddress speedup over multiprocessing python: {mp_time / bench_rs_time:,.1f}x"
)

workers available: 8
engine                             total s       addrs/s
python usaddress (1 core)             0.67        30,034
python usaddress (mp, 8 workers)      0.14       138,185
polars_usaddress                      0.04       527,662

multiprocessing speedup over single-core python: 4.6x
polars_usaddress speedup over multiprocessing python: 3.8x
